In [3]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [5]:
# 1. LOAD YOUR DATA
df = pd.read_csv("preprocessed_data1.csv")

df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Churn
0,619,0,42,2,0.00,1,1,1,101348.88,False,False,1
1,608,0,41,1,83807.86,1,0,1,112542.58,False,True,0
2,502,0,42,8,159660.80,3,1,0,113931.57,False,False,1
3,699,0,39,1,0.00,2,0,0,93826.63,False,False,0
4,850,0,43,2,125510.82,1,1,1,79084.10,False,True,0


In [11]:

## Seperating features from target 
# # X contains all columns except the target, y contains only 'Churn'
X = df.drop(columns=['Churn'])
y = df['Churn']

In [ ]:

# TRAIN-TEST SPLIT
# 'stratify=y' ensures the percentage of churned vs non-churned rows is identical in both the training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [13]:
##  FEATURE SCALING
# While Random Forest doesn't strictly require scaling, saving a scaler ensures your data pipelines remain consistent across different models.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [14]:
#  INITIALIZE AND TRAIN RANDOM FOREST
# 'class_weight="balanced"' helps the model handle class imbalance 
# (e.g., if you have far fewer claims/churns than normal records).
rf_model = RandomForestClassifier(
    n_estimators=200, 
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [15]:

# 5. INITIALIZE AND TRAIN RANDOM FOREST
# 'class_weight="balanced"' helps the model handle class imbalance 
# (e.g., if you have far fewer claims/churns than normal records).
rf_model = RandomForestClassifier(
    n_estimators=200, 
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("⏳ Training Random Forest Classifier...")
rf_model.fit(X_train_scaled, y_train)
print("✅ Training complete!\n")

⏳ Training Random Forest Classifier...
✅ Training complete!



In [17]:

# 6. EVALUATE THE MODEL
y_pred = rf_model.predict(X_test_scaled)
y_prob = rf_model.predict_proba(X_test_scaled)[:, 1]

print("=== Model Performance Evaluation ===")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

=== Model Performance Evaluation ===
ROC-AUC Score: 0.8575

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.93      0.91      1593
           1       0.66      0.56      0.60       407

    accuracy                           0.85      2000
   macro avg       0.77      0.74      0.75      2000
weighted avg       0.84      0.85      0.85      2000



In [18]:

# 7. FEATURE IMPORTANCE
# See exactly which features (like Age or Balance) influenced predictions the most
importances = rf_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\n=== Top 5 Most Important Features ===")
print(feature_importance_df.head(5).to_string(index=False))


=== Top 5 Most Important Features ===
        Feature  Importance
            Age    0.288845
  NumOfProducts    0.166435
        Balance    0.132403
EstimatedSalary    0.109248
    CreditScore    0.106919


In [20]:

# 8. SAVING MY MODEL
joblib.dump(rf_model, 'random_forest_model.pkl')
joblib.dump(scaler, 'random_forest_model_scaler.pkl')
print("\n💾 'random_forest_model.pkl' and 'random_forest_scaler.pkl' saved to disk successfully!")


💾 'random_forest_model.pkl' and 'random_forest_scaler.pkl' saved to disk successfully!
